# Question 5
## a. - e.

In [1]:
import math
import random
import textwrap
from collections import Counter, defaultdict
from typing import Sequence

from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

UNK_TOKEN = "<UNK>"

class NGramLanguageModel:
    """A word-level n-gram language model with optional Laplace smoothing."""

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("n must be at least 2 for this assignment.")

        self.n = n

        self.train_tokens: list[str] = []
        self.test_tokens: list[str] = []
        self.vocab: set[str] = set()

    def prepare_data(self, text: str) -> None:
        """Tokenize the corpus and create a sequential 90/10 train/test split.
        The vocabulary is constructed from training tokens only.
        Unseen test tokens are mapped to <UNK>.
        """
        tokens = word_tokenize(text.lower())
        split_idx = int(0.9 * len(tokens))

        self.train_tokens = tokens[:split_idx]
        self.vocab = set(self.train_tokens)
        self.vocab.add(UNK_TOKEN)

        self.test_tokens = [
            token if token in self.vocab else UNK_TOKEN
            for token in tokens[split_idx:]
        ]

    def train(self) -> None:
        """Count n-grams and the histories that have observed continuations.
        """
        if not self.train_tokens:
            raise ValueError("Call prepare_data(text) before train().")

        self.ngram_counts = Counter()
        self.history_counts = Counter()
        self.next_word_counts = defaultdict(Counter)

        for ngram in ngrams(self.train_tokens, self.n):
            history = ngram[:-1]   # The history string/tuple
            next_word = ngram[-1]  # The word following the history

            self.ngram_counts[ngram] += 1
            self.history_counts[history] += 1
            self.next_word_counts[history][next_word] += 1

    def prob(self, word: str, history: Sequence[str], smoothing=None) -> float:
        """Return P(word | history).
        Args:
            word: Candidate next token.
            history: n-1 preceding tokens.
            smoothing: None (MLE) or 'Laplace' (add-one smoothing).
        """

        history_tuple = tuple(history)
        vocab_size = len(self.vocab)

        hist_count = self.history_counts.get(history_tuple, 0)
        word_count = self.next_word_counts.get(history_tuple, {}).get(word, 0)

        if smoothing == 'Laplace':
            return (word_count + 1) / (hist_count + vocab_size)

        if hist_count == 0:
            return 0.0

        return word_count / hist_count

    def perplexity(self, tokens: Sequence[str], smoothing=None) -> float:
        """Compute perplexity using predictions with full n-1-token histories.
        """
        num_tokens = len(tokens) - (self.n - 1)
        if num_tokens <= 0:
            return float('inf')

        log_sum = 0.0 # accumulate log(prob) for numerical stability
        # skip first n - 1 tokens such that the length of history of each token in calculation is at least n - 1
        for i in range(self.n - 1, len(tokens)):
            prob = self.prob(tokens[i], tokens[i - self.n + 1: i], smoothing)
            if prob == 0:
                return float('inf')
            log_sum += math.log(prob)

        return math.exp(-log_sum / num_tokens)

    def generate(self,
        history: Sequence[str],
        max_tokens: int,
        strategy: str,
        temperature: float = 1.0,
    ) -> list[str]:
        result = []

        """Generate up to max_tokens new tokens from an initial history.
        Strategies: 'greedy', 'random', 'temperature'.
        Use MLE probabilities (no smoothing) for generation.
        Stop if the current history has no observed continuations.
        Return only the generated tokens, excluding the initial history.
        """
        if temperature <= 0.0:
            raise ValueError("Temperature must be non-negative.")

        for i in range (max_tokens):
            next_word_counts = self.next_word_counts[tuple(history)]

            if next_word_counts == {}:
                break

            if strategy == 'random':
                words, weights = zip(*next_word_counts.items())
                next_word = random.choices(words, weights=weights, k=1)[0]
            elif strategy == 'temperature':
                words, counts = zip(*next_word_counts.items())
                weights = [c ** (1.0 / temperature) for c in counts]
                next_word = random.choices(words, weights=weights, k=1)[0]
            else: # defaulting to greedy
                next_word = next_word_counts.most_common(1)[0][0]

            result.append(next_word)
            history = tuple(history[1:]) + (next_word,)

        return result

## f.

In [2]:
    text = gutenberg.raw("carroll-alice.txt")
    random.seed(42)  # For reproducible text-generation experiments

    models: dict[int, NGramLanguageModel] = {}
    stats = []

    n_values = (2, 3, 4)
    for n in n_values:
        model = NGramLanguageModel(n)
        model.prepare_data(text)
        model.train()
        models[n] = model

        stats.append({
            "n": n,
            "train_tokens": len(model.train_tokens),
            "test_tokens": len(model.test_tokens),
            "vocab_size": len(model.vocab),
            "ngram_count": len(model.ngram_counts),
            "train_mle": model.perplexity(model.train_tokens, smoothing=None),
            "test_mle": model.perplexity(model.test_tokens, smoothing=None),
            "train_laplace": model.perplexity(model.train_tokens, smoothing='Laplace'),
            "test_laplace": model.perplexity(model.test_tokens, smoothing='Laplace'),
        })

    header = f"{'n':<4} | {'Train (MLE)':<12} | {'Test (MLE)':<12} | {'Train (Laplace)':<15} | {'Test (Laplace)':<15}"

    print(header)
    print("-" * len(header))
    for s in stats:
        print(
            f"{s['n']:<4} | "
            f"{s['train_mle']:<12.4f} | "
            f"{s['test_mle']:<12.4f} | "
            f"{s['train_laplace']:<15.4f} | "
            f"{s['test_laplace']:<15.4f}"
        )


n    | Train (MLE)  | Test (MLE)   | Train (Laplace) | Test (Laplace) 
----------------------------------------------------------------------
2    | 18.8073      | inf          | 345.1561        | 508.6408       
3    | 3.3110       | inf          | 798.5102        | 1405.9644      
4    | 1.4803       | inf          | 1055.9202       | 1994.4571      


### Discussion
#### Train set
For training data, higher n is associated with lower MLE perplexity: since the model was trained on the same data, its guesses of the next token naturally improve and so it is "surprised" less as the window size increases.
However, when Laplace smoothing is used, probability of guessing the next word $w_i$ is affected by vocabulary size:
$$
    P_{laplace}(w_{i} \mid h)\ = \frac{C(hw_{i}) + 1}{C(h) + |V|}
$$

Larger n-gram size n triggers exponential drop in the amount of histories $h$, and counted sequences $hw_{i}$, such that the probability above gets closer to 0, affected by vocabulary size $|V|$. As such, perplexity increases:
$$
    P(w_{i} \mid h)\ \uparrow  \quad \implies
    \prod_{i = 1}^{N}\frac{C(hw_{i}) + 1}{C(h) + |V|} \uparrow \quad \implies
    (\prod_{i = 1}^{N}\frac{C(hw_{i}) + 1}{C(h) + |V|})^{\frac{1}{N}} \downarrow \quad \implies
    (\prod_{i = 1}^{N}\frac{C(hw_{i}) + 1}{C(h) + |V|})^{-\frac{1}{N}} \uparrow \quad \implies
    PP_{laplace}(W) \uparrow
$$

#### Test set
Under MLE the perplexity is infinite. This is caused by unobserved tokens during training (for example, token "speaker", which appears once towards the end of the text. Probability of encountering unobserved token is zero, which automatically sets perplexity to infinity.

With Laplace smoothing, the test set perplexity increases with n, similarly to train set. As $n$ increases, higher-order contexts become increasingly sparse, meaning a larger fraction of test n-grams feature contexts that were never (or rarely) seen during training. Consequently, these test n-grams are heavily penalized by the fixed vocabulary size $\vert{}V\vert{}$, driving down sequence likelihood and sharply increasing test perplexity.


## g.
see implementation in class above

## h.1

In [3]:
    generated = {}
    for (strategy, temperature) in (
            ("greedy", 1.0),
            ("random", 1.0),
            ("temperature", 0.01),
            ("temperature", 1.0),
            ("temperature", 100.0)
    ):
        result = models[3].generate(
            history=["she", "said"],
            max_tokens=100,
            strategy=strategy,
            temperature=temperature,
        )
        generated[(strategy, temperature)] = result

    # Pretty print with explicit text wrapping
    for (strategy, temperature) in generated.keys():
        temp_val = temperature if strategy == "temperature" else "N/A"
        print(f"[{strategy.upper()} | Temp: {temp_val}]")

        text = generated[(strategy, temperature)]
        # Handle both string and list-of-tokens generation outputs
        text = " ".join(str(token) for token in text)

        wrapped = textwrap.fill(text, width=80, initial_indent="  ", subsequent_indent="  ")
        print(wrapped)
        print("-" * 40)


[GREEDY | Temp: N/A]
  to herself , ' said the mock turtle , ' said the mock turtle , ' said the mock
  turtle , ' said the mock turtle , ' said the mock turtle , ' said the mock
  turtle , ' said the mock turtle , ' said the mock turtle , ' said the mock
  turtle , ' said the mock turtle , ' said the mock turtle , ' said the mock
  turtle , ' said the mock turtle , ' said the mock turtle , ' said the mock
  turtle , ' said the mock turtle , '
----------------------------------------
[RANDOM | Temp: N/A]
  to herself , ' the soldiers , or something of the door and found in it , never
  ' ) ' an arm , that it might belong to one 's own feet ! ' ' tell her to carry
  it further . so she went back to her , down , down here , and the moral of
  that is -- '' birds of a bottle marked ' poison , ' you 're mad . you 're
  talking about , and we put a stop to this mouse ? everything is so out-of-the-
  way down here , i believe .
----------------------------------------
[TEMPERATURE | Temp: 0.

**Discussion**

The temperature value 0.01 is closest to zero among [0.01, 1.0, 100.0]. When run with this temperature, the model generates sequence that very closely resembles sequence generated by the greedy model, with recurring pattern "said the mock turtle".

## h.2

In [4]:
    generated = {}
    strategy = "random"
    history = ["she", "said", "to"]
    for n in n_values:
        result = models[n].generate(
            history=history[:n - 1],
            max_tokens=100,
            strategy=strategy,
        )
        generated[n] = result

        # Pretty print with explicit text wrapping
    for n in generated.keys():
        print(f"[{strategy.upper()} | n: {n}]")

        text = generated[n]
        # Handle both string and list-of-tokens generation outputs
        text = " ".join(str(token) for token in text)

        wrapped = textwrap.fill(text, width=80, initial_indent="  ", subsequent_indent="  ")
        print(wrapped)
        print("-" * 40)


[RANDOM | n: 2]
  took up the gryphon interrupted , and , that it out to quiver all round the
  best cat seemed quite enough yet -- who i wish they were live hedgehogs , ' i
  give birthday presents to such a story for fish came nearer is you mean ,
  called after watching it 's the best butter , she came a very deeply . ' a bit
  , if it , ' alice ; and said cunning old , certainly english , and alice heard
  her toes . alice with an old magpie began again , which the door ; then
----------------------------------------
[RANDOM | n: 3]
  to herself , for the baby , it occurred to her , though , as usual . i wonder
  if i 'm a poor man , ' said alice , ' alice did not at all . i said pig , or
  she fell very slowly and sadly : -- ' you are all dry , he stole those tarts ,
  all of you to sit down without being seen , when she first saw the white
  rabbit , who seemed ready to ask his neighbour to tell me the list of the
  lobster quadrille ? ' ' i once
---------------------------------

**Discussion**

* When n=2: The text lacks global coherence and makes almost no syntactic or semantic sense. While adjacent word pairs local to the text are preserved, the connection breaks down immediately on the third word because the model's memory is restricted to just a single preceding word.

* When n=3: The output starts producing short, recognizable clauses or chunks (e.g., “for the baby , it occurred to her”, “all of you to sit down”), but still jumps abruptly between unrelated phrases.

* When n=4: The sequences gain significant fluency and grammar, producing fully recognizable, meaningful sentences straight from the book, such as:

    "cheshire cat sitting on a three-legged stool in the middle"

    "what a curious plan ! exclaimed alice"

This improvement in quality occurs because a larger n provides a longer historical context, capturing richer local syntax. However, it also reveals a limitation: because the model is trained on a single fixed text, a higher n causes it to memorize and reproduce exact verbatim chunks from the training corpus rather than generating novel, creative text.